# Comparación de todos los métodos

SEED, Laplace, Temporal Ensemble, MC Dropout, Neural Process y Deep Ensemble, más un resumen de Conformal. Evaluaciones 0 y 1 (cinco folds de test cada una) y el promedio de las dos. Los resultados se leen de la caché que crean los notebooks de cada método (`outputs/results_reports/analysis/`); si no existe, se calcula.

En los gráficos de barras, cada método tiene tres barras: evaluación 0 (claro), evaluación 1 y promedio de las dos (oscuro). Las referencias de SEED cambian levemente entre métodos porque cada uno produce sus propios errores y detecciones.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

_candidates = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next((p for p in _candidates if (p / 'configs' / 'default.yaml').exists()), Path.cwd())
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
from eeg_uq import report

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 250)


def show(figure):
    display(figure)
    plt.close(figure)

all_results = {method: report.method_results(method) for method in report.METHODS}
table = report.comparison_table(all_results)

## 1. Tabla resumen

In [ ]:
SUMMARY = ['F1 evento (%)', 'ECE clase positiva', 'NLL', 'AUROC errores: MI', 'AUPRC errores: MI', 'azar errores', 'AUROC errores: total',
           'FP vs TP AUROC MI', 'FP vs TP AUPRC MI', 'FP vs TP azar AUPRC', 'FN vs TP AUROC MI', 'FN vs TP AUPRC MI', 'FN vs TP azar AUPRC',
           '% epistémica (cerca de eventos)', 'ρ duración-incertidumbre']
for scope in report.SCOPES:
    print(scope)
    display(table.xs(scope, level='alcance')[SUMMARY].round(3))

## 2. Detección y calibración

In [ ]:
show(report.plot_method_bars(table, [('F1 evento (%)', 'F1 por evento (%)', None), ('ECE clase positiva', 'ECE clase positiva (menor es mejor)', None),
                                     ('NLL', 'NLL (menor es mejor)', None)], 'Detección y calibración'))
show(report.plot_reliability_all(all_results))

## 3. Descomposición de la incertidumbre

In [ ]:
show(report.plot_decomposition_all(table))

## 4. ¿La incertidumbre detecta los errores? (timesteps cerca de eventos)

SEED no tiene MI (sin barra en esas columnas).

In [ ]:
show(report.plot_method_bars(table, [('AUROC errores: MI', 'AUROC de errores con la MI', 0.5), ('AUPRC errores: MI', 'AUPRC de errores con la MI', 'azar errores'),
                                     ('AUROC errores: total', 'AUROC de errores con la incertidumbre total', 0.5)], 'Detección de errores cerca de eventos'))

## 5. Falsos positivos y falsos negativos

FP vs TP en el tramo predicho; FN vs TP en el tramo real. Primero con la MI, después con la incertidumbre total.

In [ ]:
show(report.plot_method_bars(table, [('FP vs TP AUROC MI', 'FP vs TP: AUROC MI', 0.5), ('FP vs TP AUPRC MI', 'FP vs TP: AUPRC MI', 'FP vs TP azar AUPRC'),
                                     ('FN vs TP AUROC MI', 'FN vs TP: AUROC MI', 0.5), ('FN vs TP AUPRC MI', 'FN vs TP: AUPRC MI', 'FN vs TP azar AUPRC')],
                             'Separación de errores por evento con la incertidumbre epistémica (MI)'))
show(report.plot_method_bars(table, [('FP vs TP AUROC total', 'FP vs TP: AUROC total', 0.5), ('FP vs TP AUPRC total', 'FP vs TP: AUPRC total', 'FP vs TP azar AUPRC'),
                                     ('FN vs TP AUROC total', 'FN vs TP: AUROC total', 0.5), ('FN vs TP AUPRC total', 'FN vs TP: AUPRC total', 'FN vs TP azar AUPRC')],
                             'Separación de errores por evento con la incertidumbre total'))

## 6. Gráficos de cajón por tipo de evento

Percentil dentro del fold (0 = la menor incertidumbre del fold, 1 = la mayor), para comparar métodos con escalas distintas. La línea punteada (0.5) es el azar; la línea roja dentro de cada caja es la media. En el título de cada panel, el AUROC de FP vs TP y de FN vs TP.

In [ ]:
for measure in ('epistémica (MI)', 'aleatoria'):
    for scope in report.SCOPES:
        show(report.plot_event_boxes_all(all_results, measure, scope))

## 7. Usos automáticos y con revisión

In [ ]:
show(report.plot_deferral_all(all_results))
show(report.plot_rejection_all(all_results))

## 8. Conformal

In [ ]:
conformal = report.conformal_results()
for alpha in (0.05, 0.10):
    print('alpha = {:.2f}'.format(alpha))
    display(report.by_evaluation(conformal[conformal['alpha'] == alpha], ['cobertura objetivo', 'cobertura total', 'cobertura husos', 'cobertura fondo', 'vacíos (abstención)']))

## Hallazgos generales

Promedio de las evaluaciones 0 y 1 (10 folds de test); los resultados son muy estables entre las dos evaluaciones.

| | F1 (%) | ECE | AUROC errores (total) | AUROC errores (MI) | FP vs TP (MI) | FN vs TP (MI) |
|---|---|---|---|---|---|---|
| SEED | 82.7 | 0.006 | 0.892 | — | — | — |
| Laplace | 82.7 | 0.006 | 0.890 | 0.811 | 0.66 | 0.19 |
| Temporal Ensemble | 82.7 | 0.006 | 0.892 | 0.815 | 0.48 | 0.31 |
| MC Dropout | 82.7 | 0.005 | 0.891 | 0.876 | 0.82 | 0.49 |
| Neural Process | 83.4 | 0.007 | 0.883 | **0.881** | **0.83** | **0.70** |
| **Deep Ensemble** | **83.7** | **0.004** | **0.899** | 0.878 | 0.78 | 0.66 |

1. **La incertidumbre de este problema es casi toda aleatoria y está en los bordes de los husos.** La parte epistémica es 1–6% en los métodos bayesianos (23% en Neural Process, con otra escala).
2. **El Deep Ensemble es el mejor método en conjunto:** mejor F1 (+1.0 punto sobre SEED), mejor calibración y mejor incertidumbre total para detectar errores.
3. **Neural Process tiene la MI más útil**, pero empeora la calibración.
4. **MC Dropout es la mejor opción sin reentrenar.** Laplace de última capa y el Temporal Ensemble aportan poco o nada.
5. **Los dos tipos de error son distintos.** Los falsos positivos son ambigüedad visible: la incertidumbre de SEED ya los marca (AUROC 0.85). Los falsos negativos son errores silenciosos: SEED casi no los delata (0.61), y solo la MI de modelos con diversidad en toda la red (Neural Process 0.70, Deep Ensemble 0.66) recupera una parte.
6. **La incertidumbre no mejora el F1 como regla automática:** entre las detecciones más inciertas nunca hay más de ~58% de FP, el mínimo necesario. Su valor está en **priorizar la revisión** (revisar el 20% más incierto captura ~60% de los FP) y en **mejorar la probabilidad promediando modelos** (Deep Ensemble).
7. **Conformal cumple su garantía en promedio, pero no dentro de los husos** (56% de cobertura con α = 0.05).